In [2]:
import sys
import os
import numpy as np
import pandas as pd

sys.path.append(os.path.abspath(".."))


In [3]:
from utils.hospital_env import HospitalEnv

data = pd.read_csv("../data/synthetic_hospital_data.csv")

env = HospitalEnv(data)


In [4]:
class QLearningAgent:
    def __init__(self, n_actions, lr=0.1, gamma=0.99, epsilon=0.1):
        self.n_actions = n_actions
        self.lr = lr
        self.gamma = gamma
        self.epsilon = epsilon
        self.q = {}

    def _state_key(self, state):
        return tuple(state.tolist())

    def select_action(self, state):
        key = self._state_key(state)
        if key not in self.q:
            self.q[key] = np.zeros(self.n_actions)

        if np.random.rand() < self.epsilon:
            return np.random.randint(self.n_actions)
        return np.argmax(self.q[key])

    def update(self, state, action, reward, next_state):
        key = self._state_key(state)

        if key not in self.q:
            self.q[key] = np.zeros(self.n_actions)

        next_key = None
        if next_state is not None:
            next_key = self._state_key(next_state)
            if next_key not in self.q:
                self.q[next_key] = np.zeros(self.n_actions)

        target = reward
        if next_state is not None:
            target += self.gamma * np.max(self.q[next_key])

        self.q[key][action] += self.lr * (target - self.q[key][action])


In [5]:
agent = QLearningAgent(
    n_actions=2,   # accept / reject
    lr=0.1,
    gamma=0.95,
    epsilon=0.1
)

N = 500


In [6]:
episode_rewards = []

for episode in range(N):
    state = env.reset()
    done = False
    total_reward = 0

    while not done:
        action = agent.select_action(state)
        next_state, reward, done = env.step(action)
        agent.update(state, action, reward, next_state)
        state = next_state
        total_reward += reward

    episode_rewards.append(total_reward)

    if episode % 50 == 0:
        print(f"Episode {episode}, Reward = {total_reward}")


Episode 0, Reward = -1067
Episode 50, Reward = 3387
Episode 100, Reward = 3370
Episode 150, Reward = 3519
Episode 200, Reward = 3566
Episode 250, Reward = 3554
Episode 300, Reward = 3495
Episode 350, Reward = 3414
Episode 400, Reward = 3438
Episode 450, Reward = 3540


In [8]:
import matplotlib.pyplot as plt

os.makedirs("../results", exist_ok=True)

plt.plot(episode_rewards)
plt.xlabel("Episode")
plt.ylabel("Total Reward")
plt.title("RL Hospital Scheduling – Training Reward")
plt.savefig("../results/reward_curve.png")
plt.close()
